# 03 Comparative Statistics

Between-group tests on the main metric block and the figures of Section 5.

**Inputs:** `all_metrics_ru.csv`, `all_metrics_foreign.csv`  
**Outputs:** `comparative_stats.csv`

> Re-runnable from this repository: the inputs are the released metric matrices.


In [ ]:
!pip install scipy statsmodels --quiet
print("✅ OK!")


# Импорты

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import mannwhitneyu, ttest_ind
from statsmodels.stats.multitest import multipletests
import math
from collections import Counter

# ── Global style ──────────────────────────────────────────────────────────
# Wong (2011) colour-blind-safe palette (also high-contrast in B&W)
C_RU  = '#0072B2'   # blue   — Russian (original)
C_FO  = '#E69F00'   # orange — Russian (translated)
H_RU  = '///'       # hatch for RU patches (diagonal lines)
H_FO  = 'xxx'       # hatch for FO patches (cross)

# ACL two-column: one figure ≈ one column = 3.35 in; two-col span = 7.0 in
COL1 = 3.35   # single column width (inches)
COL2 = 7.0    # full-width (both columns)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'font.family':          'DejaVu Sans',
    'font.size':            11,
    'axes.titlesize':       12,
    'axes.titleweight':     'bold',
    'axes.labelsize':       11,
    'xtick.labelsize':      10,
    'ytick.labelsize':      10,
    'legend.fontsize':      10,
    'figure.titlesize':     13,
    'figure.titleweight':   'bold',
    'savefig.dpi':          300,
    'figure.dpi':           120,
    'savefig.bbox':         'tight',
    'hatch.linewidth':      0.8,
    'patch.linewidth':      0.6,
})

print("✅ OK!")


# Загрузка данных


In [ ]:
# --- Paths ---------------------------------------------------------------
# BASE_PATH must point at a directory holding this notebook's input files.
# Set the KIDLIT_BASE environment variable, or edit the fallback below.
# In Google Colab: mount Drive first, then set KIDLIT_BASE to the folder there.
import os
BASE_PATH = os.environ.get("KIDLIT_BASE", "../data/") + "/"
os.makedirs(COMP_PATH, exist_ok=True)

def save(name):
    plt.savefig(COMP_PATH + name + '.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✅ {name}.png")

# Load per-corpus metrics (produced by the individual EDA notebooks)
ru = pd.read_csv(EDA_PATH + 'Графики ru/all_metrics_ru.csv',
                 sep='\t', encoding='utf-8-sig')
fo = pd.read_csv(EDA_PATH + 'Графики foreign/all_metrics_foreign.csv',
                 sep='\t', encoding='utf-8-sig')

ru['group'] = 'Russian (original)'
fo['group'] = 'Russian (translated)'

df = pd.concat([ru, fo], ignore_index=True)

print(f"Russian (original):   {len(ru)} books")
print(f"Russian (translated): {len(fo)} books")
print(f"Total:                {len(df)} books, {len(df.columns)} columns")


# Вспомогательные функции


In [ ]:
def cohens_d(a, b):
    """Effect size Cohen's d"""
    na, nb_n = len(a), len(b)
    pooled = np.sqrt(((na-1)*a.std()**2 + (nb_n-1)*b.std()**2) / (na+nb_n-2))
    return (a.mean() - b.mean()) / (pooled + 1e-10)

def compare_groups(col, label=None):
    a = ru[col].dropna()
    b = fo[col].dropna()
    t, p_t = ttest_ind(a, b, equal_var=False)
    u, p_u = mannwhitneyu(a, b, alternative='two-sided')
    d      = cohens_d(a, b)
    return {
        'metric':   label or col,
        'ru_mean':  round(a.mean(), 4),
        'fo_mean':  round(b.mean(), 4),
        'ru_std':   round(a.std(),  4),
        'fo_std':   round(b.std(),  4),
        'diff_pct': round((b.mean() - a.mean()) / (a.mean() + 1e-10) * 100, 1),
        'p_ttest':  round(p_t, 4),
        'p_mwu':    round(p_u, 4),
        'cohens_d': round(d, 3),
        'sig':      '***' if min(p_t,p_u) < 0.001
                    else '**' if min(p_t,p_u) < 0.01
                    else '*'  if min(p_t,p_u) < 0.05
                    else '',
    }

def sig_bracket(ax, x1, x2, y, p, fontsize=10):
    """Significance bracket above boxplot."""
    sym  = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    col  = '#CC0000' if sym != 'ns' else '#666666'
    span = ax.get_ylim()[1] - ax.get_ylim()[0]
    h    = span * 0.025
    ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.0, color=col)
    ax.text((x1+x2)/2, y+h*1.1, sym,
            ha='center', va='bottom', fontsize=fontsize,
            color=col, fontweight='bold')

def apply_hatch(ax, hatch_map, order):
    """Apply hatching to boxplot/bar patches for B&W accessibility."""
    # seaborn boxplot draws 2 patches per group (box body = patch)
    patches = [p for p in ax.patches if isinstance(p, matplotlib.patches.PathPatch)]
    for p, grp in zip(patches, order * (len(patches) // len(order) + 1)):
        p.set_hatch(hatch_map.get(grp, ''))
        p.set_edgecolor('black')

HATCH_MAP = {'Russian (original)': H_RU, 'Russian (translated)': H_FO}
ORDER     = ['Russian (original)', 'Russian (translated)']
PAL       = {'Russian (original)': C_RU, 'Russian (translated)': C_FO}

LEGEND_HANDLES = [
    mpatches.Patch(facecolor=C_RU, hatch=H_RU, edgecolor='black',
                   label='Russian (original)'),
    mpatches.Patch(facecolor=C_FO, hatch=H_FO, edgecolor='black',
                   label='Russian (translated)'),
]

print("✅ Helpers ready")


# Statistical summary (all metrics, with FDR correction)


In [ ]:
METRICS_MAP = {
    # Volume
    'n_words':             'Word count',
    'n_sentences':         'Sentence count',
    'pages':               'Pages',
    # Word
    'avg_word_len':        'Avg. word length',
    'avg_syllables':       'Avg. syllables/word',
    # Sentence
    'avg_sent_words':      'Avg. sentence length (words)',
    'avg_sent_chars':      'Avg. sentence length (chars)',
    # Lexical diversity
    'ttr':                 'TTR',
    'sttr':                'STTR',
    'hapax_ratio':         'Hapax ratio',
    'yule_k':              'Yule K index',
    'shannon_h':           'Shannon entropy',
    'top1000_ratio':       'Top-1000 word ratio',
    # Readability
    'flesch_ru':           'Flesch (RU)',
    'fog_index':           'FOG index',
    'ari':                 'ARI',
    'coleman_liau':        'Coleman-Liau',
    # Syntax
    'avg_tree_depth':      'Syntactic tree depth',
    'coord_sent_ratio':    'Coord. clauses ratio',
    # Grammar / morphology
    'active_voice_ratio':  'Active voice ratio',
    'past_tense_ratio':    'Past tense ratio',
    'present_tense_ratio': 'Present tense ratio',
    'future_tense_ratio':  'Future tense ratio',
    'imperative_ratio':    'Imperative mood ratio',
    'animate_noun_ratio':  'Animate noun ratio',
    'personal_pron_ratio': 'Personal pronoun ratio',
    # Part of speech
    'pos_noun_ratio':      'Nouns',
    'pos_verb_ratio':      'Verbs',
    'pos_adj_ratio':       'Adjectives',
    'pos_adv_ratio':       'Adverbs',
    'pos_propn_ratio':     'Proper nouns',
    'pos_pron_ratio':      'Pronouns',
    # Punctuation
    'quest_sent_ratio':    'Interrogative sentences',
    'excl_sent_ratio':     'Exclamatory sentences',
    'punct_diversity':     'Punctuation diversity',
}

results, _stat_cols = [], []
for col, label in METRICS_MAP.items():
    if col in ru.columns and col in fo.columns:
        results.append(compare_groups(col, label))
        _stat_cols.append(col)

stats_df = pd.DataFrame(results)

# ── FDR correction (Benjamini–Hochberg) across all metrics ────────────────
_, p_adj_bh, _, _ = multipletests(stats_df['p_ttest'], alpha=0.05, method='fdr_bh')
stats_df['p_adj_fdr'] = p_adj_bh.round(4)
stats_df['sig_adj']   = np.where(p_adj_bh < 0.001, '***',
                        np.where(p_adj_bh < 0.01,  '**',
                        np.where(p_adj_bh < 0.05,  '*',  '')))

# Lookup: column → FDR-adjusted p (significance brackets read this)
COL2ADJ = dict(zip(_stat_cols, p_adj_bh))
def padj(col):
    return float(COL2ADJ.get(col, 1.0))

stats_df.to_csv(COMP_PATH + 'comparative_stats.csv',
                sep='\t', index=False, encoding='utf-8-sig')

print(f"\n{'Metric':<30} {'orig':>8} {'transl':>8} {'Δ%':>7} {'p_raw':>8} {'p_FDR':>8} {'d':>7} {'sig':>4}")
print("─"*88)
for _, r in stats_df.iterrows():
    print(f"{r['metric']:<30} {r['ru_mean']:>8.3f} {r['fo_mean']:>8.3f} "
          f"{r['diff_pct']:>+6.1f}% {r['p_ttest']:>8.4f} {r['p_adj_fdr']:>8.4f} "
          f"{r['cohens_d']:>7.3f} {r['sig_adj']:>4}")

print(f"\n  Significant before FDR (p<0.05): {(stats_df['sig']!='').sum()} / {len(stats_df)}")
print(f"  Significant after  FDR (p<0.05): {(stats_df['sig_adj']!='').sum()} / {len(stats_df)}")
print(f"\n✅ Saved → comparative_stats.csv")

# Text volume


In [ ]:
vol_metrics = [
    ('n_words',     'Word count'),
    ('n_sentences', 'Sentence count'),
    ('pages',       'Pages'),
]
vol_metrics = [(c, l) for c, l in vol_metrics if c in df.columns]
n = len(vol_metrics)

fig, axes = plt.subplots(1, n, figsize=(COL2, 3.2))
if n == 1: axes = [axes]
fig.suptitle('Text volume', y=1.02)

for ax, (col, label) in zip(axes, vol_metrics):
    bp = sns.boxplot(data=df, x='group', y=col, ax=ax,
                     palette=PAL, order=ORDER, width=0.5,
                     linewidth=1.2, fliersize=3)
    apply_hatch(ax, HATCH_MAP, ORDER)
    sns.stripplot(data=df, x='group', y=col, ax=ax,
                  color='black', order=ORDER,
                  size=3, alpha=0.45, jitter=True)

    lo, hi = ax.get_ylim(); rng = hi - lo
    for j, grp in enumerate(ORDER):
        m = df[df['group'] == grp][col].mean()
        ax.text(j, hi - rng*0.04, f'M={m:.0f}',
                ha='center', va='top', fontsize=9, fontweight='bold',
                color='white',
                bbox=dict(boxstyle='round,pad=0.15',
                          facecolor=PAL[grp], alpha=0.85, edgecolor='none'))
    sig_bracket(ax, 0, 1, hi - rng*0.12, padj(col))

    ax.set_title(label, pad=4)
    ax.set_xlabel('')
    ax.set_xticklabels(['Orig.', 'Transl.'])
    ax.set_ylabel('')

axes[0].set_ylabel('Count')
fig.legend(handles=LEGEND_HANDLES, loc='upper center',
           bbox_to_anchor=(0.5, -0.08), ncol=2, frameon=False)
plt.tight_layout()
save('plot_comp_volume')

# Lexical diversity


In [ ]:
lex_metrics = [
    ('ttr',           'TTR'),
    ('sttr',          'STTR'),
    ('hapax_ratio',   'Hapax ratio'),
    ('shannon_h',     'Shannon entropy'),
    ('yule_k',        'Yule K index'),
    ('top1000_ratio', 'Top-1000 ratio'),
]
lex_metrics = [(c, l) for c, l in lex_metrics if c in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(COL2, 5.5))
fig.suptitle('Lexical diversity', y=1.01)

for ax, (col, label) in zip(axes.flat, lex_metrics):
    vp = sns.violinplot(data=df, x='group', y=col, ax=ax,
                        palette=PAL, order=ORDER,
                        inner=None, linewidth=0.8, cut=0.5)
    # Hatch the violin bodies
    for coll, grp in zip(ax.collections, ORDER):
        coll.set_hatch(HATCH_MAP[grp])
        coll.set_edgecolor('black')
        coll.set_linewidth(0.6)
    # Inner box (quartiles)
    sns.boxplot(data=df, x='group', y=col, ax=ax,
                palette=PAL, order=ORDER,
                width=0.12, linewidth=1.0,
                fliersize=0, showfliers=False)

    a = ru[col].dropna(); b = fo[col].dropna()
    d = cohens_d(a, b)

    lo, hi = ax.get_ylim(); rng = hi - lo
    for j, vals in enumerate([a, b]):
        ax.text(j, lo + rng*0.04,
                f'M={vals.mean():.3f}',
                ha='center', fontsize=8, fontweight='bold', color='white',
                bbox=dict(boxstyle='round,pad=0.15',
                          facecolor='#333333', alpha=0.75, edgecolor='none'))
    sig_bracket(ax, 0, 1, hi - rng*0.1, padj(col), fontsize=9)
    ax.set_title(f'{label}  d={d:+.2f}', fontsize=10, pad=3)
    ax.set_xlabel(''); ax.set_ylabel('')
    ax.set_xticklabels(['Orig.', 'Transl.'])

for j in range(len(lex_metrics), len(axes.flat)):
    axes.flat[j].set_visible(False)

fig.legend(handles=LEGEND_HANDLES, loc='upper center',
           bbox_to_anchor=(0.5, -0.04), ncol=2, frameon=False)
plt.tight_layout()
save('plot_comp_lexical_diversity')

# Syntactic complexity


In [ ]:
syn_metrics = [
    ('avg_sent_words',   'Sent. length (words)'),
    ('avg_tree_depth',   'Syntactic tree depth'),
    ('coord_sent_ratio', 'Coord. clauses'),
    ('avg_sent_chars',   'Sent. length (chars)'),
    ('n_long_sents',     'Long sents. (≥20 w.)'),
    ('n_short_sents',    'Short sents. (≤5 w.)'),
]
syn_metrics = [(c, l) for c, l in syn_metrics if c in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(COL2, 5.5))
fig.suptitle('Syntactic complexity', y=1.01)

for ax, (col, label) in zip(axes.flat, syn_metrics):
    bp = sns.boxplot(data=df, x='group', y=col, ax=ax,
                     palette=PAL, order=ORDER, width=0.5,
                     linewidth=1.0, fliersize=3)
    apply_hatch(ax, HATCH_MAP, ORDER)
    sns.stripplot(data=df, x='group', y=col, ax=ax,
                  color='black', order=ORDER,
                  size=2.5, alpha=0.4, jitter=True)
    sig_bracket(ax, 0, 1, ax.get_ylim()[1]*0.88, padj(col), fontsize=9)
    ax.set_title(label, fontsize=10, pad=3)
    ax.set_xlabel('')
    ax.set_xticklabels(['Orig.', 'Transl.'])

for j in range(len(syn_metrics), len(axes.flat)):
    axes.flat[j].set_visible(False)

fig.legend(handles=LEGEND_HANDLES, loc='upper center',
           bbox_to_anchor=(0.5, -0.04), ncol=2, frameon=False)
plt.tight_layout()
save('plot_comp_syntax')

# Readability


In [ ]:
read_metrics = [
    ('flesch_ru',    'Flesch (RU)\n↑ easier'),
    ('fog_index',    'FOG\n↑ harder'),
    ('ari',          'ARI\n↑ harder'),
    ('coleman_liau', 'Coleman-Liau\n↑ harder'),
]
read_metrics = [(c, l) for c, l in read_metrics if c in df.columns]
n = len(read_metrics)

fig, axes = plt.subplots(1, n, figsize=(COL2, 3.4))
if n == 1: axes = [axes]
fig.suptitle('Readability (complexity indices)', y=1.02)

for ax, (col, label) in zip(axes, read_metrics):
    sns.boxplot(data=df, x='group', y=col, ax=ax,
                palette=PAL, order=ORDER, width=0.5,
                linewidth=1.0, fliersize=3)
    apply_hatch(ax, HATCH_MAP, ORDER)
    sns.stripplot(data=df, x='group', y=col, ax=ax,
                  color='black', order=ORDER,
                  size=3, alpha=0.4, jitter=True)
    sig_bracket(ax, 0, 1, ax.get_ylim()[1]*0.88, padj(col), fontsize=9)
    ax.set_title(label, fontsize=10, pad=3)
    ax.set_xlabel('')
    ax.set_xticklabels(['Orig.', 'Transl.'])
    ax.set_ylabel('')

axes[0].set_ylabel('Index value')
fig.legend(handles=LEGEND_HANDLES, loc='upper center',
           bbox_to_anchor=(0.5, -0.08), ncol=2, frameon=False)
plt.tight_layout()
save('plot_comp_readability')

# Summary


In [ ]:
print("\n" + "═"*60)
print("  COMPARATIVE EDA — SUMMARY")
print("═"*60)
print(f"  Russian (original):   {len(ru)} books")
print(f"  Russian (translated): {len(fo)} books")
print(f"  Metrics compared:     {len(stats_df)}")
print(f"  Significant after FDR (p<0.05): {(stats_df['sig_adj']!='').sum()} / {len(stats_df)}")
print(f"\n  Table:")
print(f"     comparative_stats.csv  (raw + FDR-adjusted p-values)")
print(f"\n  Figures:")
for p in ['plot_comp_volume', 'plot_comp_lexical_diversity',
          'plot_comp_syntax', 'plot_comp_readability']:
    print(f"     {p}.png")
